<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Histogram**


Estimated time needed: **45** minutes


In this lab, you will focus on the visualization of data. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


In this lab, you will perform the following:


- Visualize the distribution of data using histograms.

- Visualize relationships between features.

- Explore data composition and comparisons.


## Demo: Working with database


#### Download the database file.


In [ ]:
!wget -O survey-data.sqlite https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QR9YeprUYhOoLafzlLspAw/survey-results-public.sqlite

#### Install the required libraries and import them


In [ ]:
!pip install pandas

In [ ]:
!pip install matplotlib

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

#### Connect to the SQLite database


In [ ]:
conn = sqlite3.connect('survey-data.sqlite')

## Demo: Basic SQL queries

**Demo 1: Count the number of rows in the table**


In [ ]:
QUERY = "SELECT COUNT(*) FROM main"
df = pd.read_sql_query(QUERY, conn)
print(df)


**Demo 2: List all tables**


In [ ]:
QUERY = """
SELECT name as Table_Name 
FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


**Demo 3: Group data by age**


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count 
FROM main 
GROUP BY Age 
ORDER BY Age
"""
df_age = pd.read_sql_query(QUERY, conn)
print(df_age)


## Hands-on Lab: Visualizing Data with Histograms


### 1. Visualizing the distribution of data (Histograms)


**1.1 Histogram of `CompTotal` (Total Compensation)**


Objective: Plot a histogram of `CompTotal` to visualize the distribution of respondents' total compensation.


In [ ]:
# ---- Helpers used throughout this lab ----
import numpy as np

# Age is stored as text ranges, so map each range to an approximate number (its midpoint)
AGE_MAP = {
    'Under 18 years old': 17, '18-24 years old': 21, '25-34 years old': 29.5,
    '35-44 years old': 39.5, '45-54 years old': 49.5, '55-64 years old': 59.5,
    '65 years or older': 70,
}
AGE_ORDER = list(AGE_MAP.keys())

# TimeSearching / TimeAnswering are text ranges -> approximate minutes per day
TIME_MAP = {
    'Less than 15 minutes a day': 7.5, '15-30 minutes a day': 22.5,
    '30-60 minutes a day': 45, '60-120 minutes a day': 90, 'Over 120 minutes a day': 150,
}

def years_to_num(series):
    """YearsCodePro is text: turn 'Less than 1 year' / 'More than 50 years' into numbers."""
    return pd.to_numeric(series.replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
                         errors='coerce')

def split_counts(series, top_n=None):
    """Count items in a ';'-separated multi-answer column."""
    counts = series.dropna().str.split(';').explode().str.strip().value_counts()
    return counts.head(top_n) if top_n else counts

# ---- 1.1 Histogram of CompTotal ----
comp = pd.read_sql_query("SELECT CompTotal FROM main WHERE CompTotal IS NOT NULL", conn)
comp['CompTotal'] = pd.to_numeric(comp['CompTotal'], errors='coerce')
limit = comp['CompTotal'].quantile(0.99)   # trim the extreme top 1% so the shape is visible

plt.figure(figsize=(10, 5))
plt.hist(comp.loc[comp['CompTotal'] <= limit, 'CompTotal'], bins=50, color='steelblue', edgecolor='black')
plt.title('Distribution of CompTotal (up to 99th percentile)')
plt.xlabel('Total compensation')
plt.ylabel('Number of respondents')
plt.show()

**1.2 Histogram of YearsCodePro (Years of Professional Coding Experience)**


Objective: Plot a histogram of `YearsCodePro` to analyze the distribution of coding experience among respondents.


In [ ]:
# ---- 1.2 Histogram of YearsCodePro ----
yc = pd.read_sql_query("SELECT YearsCodePro FROM main WHERE YearsCodePro IS NOT NULL", conn)
yc['YearsCodePro_num'] = years_to_num(yc['YearsCodePro'])

plt.figure(figsize=(10, 5))
plt.hist(yc['YearsCodePro_num'].dropna(), bins=range(0, 52, 2), color='seagreen', edgecolor='black')
plt.title('Distribution of Years of Professional Coding')
plt.xlabel('YearsCodePro')
plt.ylabel('Number of respondents')
plt.show()

### 2. Visualizing Relationships in Data


**2.1 Histogram Comparison of `CompTotal` by `Age` Group**


Objective: Use histograms to compare the distribution of CompTotal across different Age groups.


In [ ]:
# ---- 2.1 CompTotal by Age group ----
ca = pd.read_sql_query("SELECT Age, CompTotal FROM main WHERE CompTotal IS NOT NULL AND Age IS NOT NULL", conn)
ca['CompTotal'] = pd.to_numeric(ca['CompTotal'], errors='coerce')
limit = ca['CompTotal'].quantile(0.99)
ca = ca[ca['CompTotal'] <= limit]

plt.figure(figsize=(12, 6))
for group in ['18-24 years old', '25-34 years old', '35-44 years old', '45-54 years old', '55-64 years old']:
    plt.hist(ca.loc[ca['Age'] == group, 'CompTotal'], bins=40, alpha=0.45, label=group)
plt.title('CompTotal Distribution by Age Group')
plt.xlabel('Total compensation')
plt.ylabel('Number of respondents')
plt.legend()
plt.show()

**2.2 Histogram of TimeSearching for Different Age Groups**


Objective: Use histograms to explore the distribution of `TimeSearching` (time spent searching for information) for respondents across different age groups.


In [ ]:
# ---- 2.2 TimeSearching for different age groups ----
ts = pd.read_sql_query("SELECT Age, TimeSearching FROM main WHERE TimeSearching IS NOT NULL AND Age IS NOT NULL", conn)
ts['Time_minutes'] = ts['TimeSearching'].map(TIME_MAP)

bins = [0, 15, 30, 60, 120, 180]
plt.figure(figsize=(12, 6))
for group in ['18-24 years old', '25-34 years old', '35-44 years old', '45-54 years old']:
    plt.hist(ts.loc[ts['Age'] == group, 'Time_minutes'], bins=bins, alpha=0.45, label=group, density=True)
plt.title('Time Spent Searching for Information by Age Group')
plt.xlabel('Approx. minutes per day (bins: <15, 15-30, 30-60, 60-120, 120+)')
plt.ylabel('Share of respondents (density)')
plt.legend()
plt.show()

### 3. Visualizing the Composition of Data


**3.1 Histogram of Most Desired Databases (`DatabaseWantToWorkWith`)**


Objective: Visualize the most desired databases for future learning using a histogram of the top 5 databases.


In [ ]:
# ---- 3.1 Top 5 most desired databases ----
db = pd.read_sql_query("SELECT DatabaseWantToWorkWith FROM main WHERE DatabaseWantToWorkWith IS NOT NULL", conn)
top5 = split_counts(db['DatabaseWantToWorkWith'], 5)

plt.figure(figsize=(10, 5))
plt.bar(top5.index, top5.values, color='steelblue', edgecolor='black')
plt.title('Top 5 Databases Respondents Want to Work With')
plt.xlabel('Database')
plt.ylabel('Number of respondents')
plt.show()

**3.2 Histogram of Preferred Work Locations (`RemoteWork`)**


Objective: Use a histogram to explore the distribution of preferred work arrangements (`remote work`).


In [ ]:
# ---- 3.2 Preferred work arrangements (RemoteWork) ----
rw = pd.read_sql_query("""
SELECT RemoteWork, COUNT(*) AS count
FROM main
WHERE RemoteWork IS NOT NULL
GROUP BY RemoteWork
ORDER BY count DESC
""", conn)

plt.figure(figsize=(10, 5))
plt.bar(rw['RemoteWork'], rw['count'], color='orange', edgecolor='black')
plt.title('Distribution of Work Arrangements')
plt.ylabel('Number of respondents')
plt.xticks(rotation=15)
plt.show()

### 4. Visualizing Comparison of Data


**4.1 Histogram of Median CompTotal for Ages 45 to 60**


Objective: Plot the histogram for `CompTotal` within the age group 45 to 60 to analyze compensation distribution among mid-career respondents.


In [ ]:
# ---- 4.1 CompTotal for ages 45 to 60 ----
mid = pd.read_sql_query("""
SELECT CompTotal
FROM main
WHERE Age IN ('45-54 years old', '55-64 years old') AND CompTotal IS NOT NULL
""", conn)
mid['CompTotal'] = pd.to_numeric(mid['CompTotal'], errors='coerce')
limit = mid['CompTotal'].quantile(0.99)
print("Median CompTotal (ages 45-60):", mid['CompTotal'].median())

plt.figure(figsize=(10, 5))
plt.hist(mid.loc[mid['CompTotal'] <= limit, 'CompTotal'], bins=40, color='purple', edgecolor='black')
plt.axvline(mid['CompTotal'].median(), color='red', linestyle='--', label='Median')
plt.title('CompTotal for Respondents Aged 45 to 60')
plt.xlabel('Total compensation')
plt.ylabel('Number of respondents')
plt.legend()
plt.show()

**4.2 Histogram of Job Satisfaction (`JobSat`) by YearsCodePro**


Objective: Plot the histogram for `JobSat` scores based on respondents' years of professional coding experience.


In [ ]:
# ---- 4.2 JobSat by years of professional coding ----
js = pd.read_sql_query("SELECT JobSat, YearsCodePro FROM main WHERE JobSat IS NOT NULL AND YearsCodePro IS NOT NULL", conn)
js['JobSat'] = pd.to_numeric(js['JobSat'], errors='coerce')
js['ExpGroup'] = pd.cut(years_to_num(js['YearsCodePro']), bins=[-1, 5, 10, 20, 100],
                        labels=['0-5 yrs', '6-10 yrs', '11-20 yrs', '20+ yrs'])

plt.figure(figsize=(12, 6))
for group in js['ExpGroup'].cat.categories:
    plt.hist(js.loc[js['ExpGroup'] == group, 'JobSat'], bins=range(0, 12), alpha=0.45,
             label=str(group), density=True, align='left')
plt.title('Job Satisfaction by Years of Professional Coding')
plt.xlabel('JobSat (0-10)')
plt.ylabel('Share of respondents (density)')
plt.legend()
plt.show()

### Final step: Close the database connection


Once you've completed the lab, make sure to close the connection to the SQLite database:



In [ ]:
conn.close()

### Summary


In this lab, you used histograms to visualize various aspects of the dataset, focusing on:

- Distribution of compensation, coding experience, and work hours.

- Relationships in compensation across age groups and work status.

- Composition of data by desired databases and work environments.

- Comparisons of job satisfaction across years of experience.

Histograms helped reveal patterns and distributions in the data, enhancing your understanding of developer demographics and preferences.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


Copyright © IBM Corporation. All rights reserved.
